**This notebook was entirely AI GENERATED by GLM 5.2 using opencode (prompted by GN)**

# Classification stability across retraining

Compare two index variants to see how the per-domain top-N entries change after retraining the semantic-search classifier:

- **baseline (A)** = `2026-07-18`
- **retrained (B)** = `2026-07-18retrained`

Source data: the variant `index.json` files (table orientation), restricted to edition 7. Each entry carries a `semantic_search-label` (its assigned domain) and a `semantic_search-score` (similarity to that domain).

For each `N` in `{10, 100, 1000, 10000}` and each of the 8 domains `D`, let `T_A` be the top-N entries of `D` in A and `T_B` the top-N entries of `D` in B (ties broken by `index` ascending). We count:

1. **No longer in top N** — `|T_A \ T_B|`: entries that were in the top N of domain `D` in A but are not in the top N of the *same* domain `D` in B. An entry that switched domain is automatically counted here.
2. **Different domain** — `|{e in T_A : label_B(e) != D}|`: entries of `T_A` whose assigned domain in B is no longer `D`.

Because the top-N of `D` in B only contains entries still labelled `D`, the second metric is always a subset of the first. At `N = 10000` (>= every domain's size) the two metrics coincide: leaving the top N of `D` is equivalent to no longer being labelled `D`.

In [1]:
VERSIONS = ['2026-07-18', '2026-07-18retrained']
N_VALUES = [10, 100, 500, 1000]
EDITION = 7

## Load the two indexes

Read each variant's `index.json` (table orientation), keep edition 7 and only labelled entries. With `orient='table'` and `primaryKey=['index']`, pandas sets the entry id as the DataFrame index.

In [2]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))
from helpers import settings

indexes = {}
for variant in VERSIONS:
    path = Path(settings.DATA_PATH, variant, 'index.json')
    df = pd.read_json(path, orient='table')
    df = df[df['edition'] == EDITION]
    df = df[df['semantic_search-label'].notna()]
    indexes[variant] = df
    print(f'{variant}: {len(df)} labelled entries, {df["semantic_search-label"].nunique()} domains')

2026-07-18: 21118 labelled entries, 8 domains
2026-07-18retrained: 21118 labelled entries, 8 domains


In [3]:
df_a = indexes[VERSIONS[0]]
df_b = indexes[VERSIONS[1]]

domains = sorted(df_a['semantic_search-label'].unique())
print(f'Domains ({len(domains)}): {domains}')
print(f'Common index keys: {len(set(df_a.index) & set(df_b.index))}')
print(f'Only in A: {len(set(df_a.index) - set(df_b.index))}, only in B: {len(set(df_b.index) - set(df_a.index))}')

print('\nDomain sizes:')
sizes = pd.DataFrame({VERSIONS[0]: df_a['semantic_search-label'].value_counts(),
                      VERSIONS[1]: df_b['semantic_search-label'].value_counts(),
                      "Difference": df_a['semantic_search-label'].value_counts() - df_b['semantic_search-label'].value_counts()
                     }).reindex(domains)
sizes

Domains (8): ['civil_history', 'fine_arts', 'natural_history', 'natural_science', 'philosophy', 'sacred_history', 'theology', 'useful_arts']
Common index keys: 21118
Only in A: 0, only in B: 0

Domain sizes:


,2026-07-18,2026-07-18retrained,Difference
semantic_search-label,,,
civil_history,1675,1726,-51
fine_arts,1477,1581,-104
natural_history,2502,2623,-121
natural_science,2366,2403,-37
philosophy,1944,2079,-135
sacred_history,3595,3428,167
theology,1124,1061,63
useful_arts,6435,6217,218


## Build the per-domain top-N sets

For each variant and domain, the top-N entries by `semantic_search-score` (ties broken by `index` ascending for determinism). `sacred_history` is the only domain with tied scores, so the tie-break is relevant there.

In [4]:
def top_n(df, domain, n):
    """Return the set of entry ids in the top-n of `domain` by score (ties broken by index asc)."""
    sub = df[df['semantic_search-label'] == domain].reset_index()
    ordered = sub.sort_values(['semantic_search-score', 'index'], ascending=[False, True])
    return set(ordered.head(n)['index'])

top_n_sets = {variant: {d: {n: top_n(df, d, n) for n in N_VALUES} for d in domains}
              for variant, df in [(VERSIONS[0], df_a), (VERSIONS[1], df_b)]}

label_b = df_b['semantic_search-label'].to_dict()

print('Top-N sets built for', len(domains), 'domains x', len(N_VALUES), 'N values x', len(VERSIONS), 'variants')

Top-N sets built for 8 domains x 4 N values x 2 variants


## Compute the metrics

For each `N` and domain `D`:
- `no_longer` = `|T_A \ T_B|` (in top N of D in A but not in top N of D in B)
- `diff_domain` = entries of `T_A` whose label in B is no longer `D`

The summary table totals across all 8 domains (denominator = `8 x N`).

In [5]:
summary_rows = []
per_domain_rows = []

for n in N_VALUES:
    no_longer_total = 0
    diff_domain_total = 0
    for d in domains:
        t_a = top_n_sets[VERSIONS[0]][d][n]
        t_b = top_n_sets[VERSIONS[1]][d][n]
        no_longer = len(t_a - t_b)
        diff_domain = sum(1 for e in t_a if label_b[e] != d)
        no_longer_total += no_longer
        diff_domain_total += diff_domain
        per_domain_rows.append({
            'domain': d,
            'N': n,
            'topN_in_A': len(t_a),
            'no_longer_in_topN': no_longer,
            'different_domain': diff_domain,
        })
    # denom = min(len(domains) * n, len(df_a))
    denom = len(domains) * n
    summary_rows.append({
        'N': n,
        'no_longer_in_topN': no_longer_total,
        'no_longer_%': round(no_longer_total / denom * 100, 1),
        'different_domain': diff_domain_total,
        'different_domain_%': round(diff_domain_total / denom * 100, 1),
    })

summary = pd.DataFrame(summary_rows).set_index('N')
per_domain = pd.DataFrame(per_domain_rows)
summary

,no_longer_in_topN,no_longer_%,different_domain,different_domain_%
N,,,,
10,24,30.0,0,0.0
100,217,27.1,11,1.4
500,830,20.8,214,5.3
1000,1567,19.6,778,9.7


## Per-domain breakdown

In [6]:
per_domain_pivot = per_domain.pivot(index='domain', columns='N',
                                    values=['no_longer_in_topN', 'different_domain'])
per_domain_pivot

no_longer_in_topN                different_domain            \
N                            10   100  500  1000             10   100  500    
domain                                                                        
civil_history                   2   28  114  207                0    3   37   
fine_arts                       1   20   70  133                0    0   15   
natural_history                 3   40  145  239                0    3   31   
natural_science                 5   27  129  190                0    1   25   
philosophy                      2   15   71  159                0    0   13   
sacred_history                  6   32  102  189                0    0   23   
theology                        2   24   96  264                0    4   70   
useful_arts                     3   31  103  186                0    0    0   

                      
N               1000  
domain                
civil_history    115  
fine_arts         81  
natural_history  100  
natural_science   68  
philosophy        76  
sacred_history    76  
theology         261  
useful_arts        1